# Segmentation người đi bộ: U-Net tự dựng, encoder pretrain, và một hàm serve không lệch

Bài Intermediate 03 đo YOLO11n-seg chưa từng thấy Penn-Fudan. Ở đây bạn tự train một mạng **semantic
segmentation** (mỗi pixel: nền hay người) trên chính Penn-Fudan, với cùng cách chia train/val như bài Advanced 01
(136 ảnh train, 34 ảnh val), rồi so có đối chứng: U-Net train từ đầu, U-Net với encoder ResNet-18 pretrain ImageNet,
và YOLO11n-seg zero-shot. Cuối cùng là một hàm `predict_mask` nhận ảnh **BGR** cỡ bất kỳ từ OpenCV (như khung hình
webcam) và trả mask đúng độ phân giải gốc: chỗ mà train/serve skew hay xảy ra nhất.

**Cách làm bài**
- Hàm và lớp dưới đây là **API test gọi** (tên và thứ tự tham số giữ nguyên; được thêm tham số keyword có mặc định).
  Phần còn lại (vòng train, loss tổng, encoder pretrain, lịch lr, số epoch, đánh giá) là thiết kế của bạn.
- Cell gắn tag `explore` chạy thí nghiệm thật; test bỏ qua chúng. Chọn kernel `ai-gpu` (xem `setup/gpu/README.md`).
- Chấm: `uv run pytest tracks/ai-engineer/modules/03-computer-vision/exercises/advanced/02-unet-segmentation-pennfudan -q`

In [ ]:
# explore: giới hạn luồng CPU, tìm gốc repo, kiểm Penn-Fudan (53 MB, tải một lần bằng env gốc)
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")   # đặt TRƯỚC khi import numpy/torch
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
PF_ROOT = ROOT / "data/raw/penn-fudan"
if not (PF_ROOT.is_dir() and any(PF_ROOT.iterdir())):
    # kernel ai-gpu không có thư viện tải dữ liệu: khi đó chạy ở terminal `uv run python data/download_dataset.py penn-fudan`
    sys.path.insert(0, str(ROOT))
    from data.download_dataset import ensure

    ensure("penn-fudan")
PF = PF_ROOT / "PennFudanPed"
print(PF, sorted(p.name for p in PF.iterdir()))

In [ ]:
import math
import random
import time
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def load_pair(root: Path, name: str) -> tuple[np.ndarray, np.ndarray]:
    """Có sẵn. Ảnh RGB uint8 (H, W, 3) và mask nhị phân uint8 (H, W): 1 là người (mọi instance), 0 là nền."""
    root = Path(root)
    image = np.asarray(Image.open(root / "PNGImages" / f"{name}.png").convert("RGB"))
    mask = (np.asarray(Image.open(root / "PedMasks" / f"{name}_mask.png")) > 0).astype(np.uint8)
    return image, mask


def split_names(names: list[str], val_frac: float = 0.2, seed: int = 0) -> tuple[list[str], list[str]]:
    """Có sẵn. Cùng cách chia với prepare_data.py của bài Advanced 01: val = sample ngẫu nhiên trên danh sách đã sắp."""
    names = sorted(names)
    val = sorted(random.Random(seed).sample(names, round(len(names) * val_frac)))
    return [n for n in names if n not in set(val)], val

## 1. Transform chung cho ảnh và mask

`joint_transform(image, mask, size=256, train=False, generator=None) -> (x, m)`:

- `image` RGB uint8 `(H, W, 3)`, `mask` uint8 `(H, W)` giá trị 0/1. Trả `x` float32 `(3, size, size)` đã chia 255 và
  chuẩn hoá theo `IMAGENET_MEAN`/`IMAGENET_STD`, `m` int64 `(size, size)` chỉ gồm 0 và 1.
- **eval** (`train=False`): resize thẳng về `size × size` (ảnh: nội suy bilinear; mask: **nearest**, không được sinh
  giá trị lạ). Tất định.
- **train**: thêm augmentation hình học ngẫu nhiên (ít nhất một random crop/scale và lật ngang) áp **cùng một phép**
  cho ảnh và mask, rồi resize như eval. Mọi số ngẫu nhiên lấy từ `generator` (`torch.Generator`): cùng seed, cùng kết
  quả. Augmentation màu (nếu có) chỉ áp cho ảnh.

In [ ]:
def joint_transform(image: np.ndarray, mask: np.ndarray, size: int = 256, train: bool = False,
                    generator: torch.Generator | None = None) -> tuple[torch.Tensor, torch.Tensor]:
    # TODO
    raise NotImplementedError

## 2. U-Net

`UNet(in_ch=3, num_classes=2, base=32)`: encoder 4 lần giảm mẫu (mỗi mức hai conv 3×3 + BatchNorm + ReLU, số kênh
`base, 2·base, 4·base, 8·base`, đáy `16·base`), decoder đối xứng: upsample ×2, **nối (concat) feature cùng độ phân
giải từ encoder**, hai conv 3×3; cuối cùng conv 1×1 ra `num_classes` kênh. Input `(B, in_ch, H, W)` với `H`, `W` chia
hết cho 16, output logits `(B, num_classes, H, W)`. Chọn upsample bằng `ConvTranspose2d` hay bilinear + conv là
quyết định của bạn (ghi vào báo cáo).

Encoder pretrain (ResNet-18 ImageNet của torchvision làm nhánh mã hoá, decoder tự viết) là lớp riêng của bạn, không
bị test; báo cáo phải so nó với `UNet` từ đầu.

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_ch: int = 3, num_classes: int = 2, base: int = 32):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO
        raise NotImplementedError

## 3. Dice loss

`dice_loss(logits, target, eps=1.0) -> tensor vô hướng`: soft Dice của lớp người (lớp 1) tính trên **cả batch**
(cộng mọi pixel mọi ảnh trước khi chia), với $$p$$ là xác suất softmax của lớp 1 và $$t$$ là mask 0/1:

$$
\mathcal{L}_{Dice} = 1 - \frac{2\sum p\,t + \varepsilon}{\sum p + \sum t + \varepsilon}
$$

`logits` shape `(B, 2, H, W)`, `target` int64 `(B, H, W)`. Khả vi theo `logits`.

In [ ]:
def dice_loss(logits: torch.Tensor, target: torch.Tensor, eps: float = 1.0) -> torch.Tensor:
    # TODO
    raise NotImplementedError

## 4. Hàm serve

`predict_mask(model, image_bgr, size=256, device="cpu") -> np.ndarray`: nhận một khung hình **BGR** uint8 `(H, W, 3)`
như `cv2.imread` hay `cv2.VideoCapture` trả về, cỡ bất kỳ; tiền xử lý **giống hệt** lúc eval (dùng lại
`joint_transform`), chạy model ở chế độ `eval()` không gradient, nội suy **logits** về đúng `(H, W)` của ảnh gốc
(bilinear, `align_corners=False`) rồi mới lấy lớp có điểm cao nhất. Trả mask uint8 `(H, W)` giá trị 0/1. Trả model về đúng chế độ (train/eval) như trước khi gọi.

In [ ]:
@torch.no_grad()
def predict_mask(model: nn.Module, image_bgr: np.ndarray, size: int = 256, device: str = "cpu") -> np.ndarray:
    # TODO
    raise NotImplementedError

## 5. Thí nghiệm của bạn

Gợi ý thứ tự: chia dữ liệu bằng `split_names`, vẽ vài cặp (ảnh, mask) sau `joint_transform(train=True)` để chắc
mask khớp ảnh, overfit một batch 4 ảnh, rồi mới train thật. Đánh giá trên 34 ảnh val **ở độ phân giải gốc** qua
`predict_mask`, mIoU tính trên tổng pixel cả tập (như `iou_from_confusion` của bài Intermediate 03).

In [ ]:
# explore: chia dữ liệu như bài Advanced 01, nạp vào RAM (170 ảnh, khoảng 150 MB)
names = sorted(p.stem for p in (PF / "PNGImages").glob("*.png"))
train_names, val_names = split_names(names)
train_pairs = [load_pair(PF, n) for n in train_names]
val_pairs = [load_pair(PF, n) for n in val_names]
print(len(train_pairs), "ảnh train,", len(val_pairs), "ảnh val; val đầu:", val_names[:3], "| device", DEVICE)

In [ ]:
# explore: TODO thí nghiệm của bạn: kiểm transform bằng hình, overfit một batch, train UNet từ đầu,
# train encoder ResNet-18 pretrain, đo mIoU val ở độ phân giải gốc, so với YOLO11n-seg zero-shot trên cùng 34 ảnh,
# đo ms/ảnh của predict_mask, lưu hình cho REPORT.md